# Smoke Pipeline Run (Fast)

A short end-to-end check of the full SFT -> DPO flow.

Use this notebook to validate that scripts, paths, and environment work before a long run.

## 0) Environment setup (one-time)

If this machine is already prepared, skip this cell.

In [1]:
import sys
print(sys.executable)

/workspace/thesis-llm-alignment/.venv/bin/python


In [2]:
from pathlib import Path
import subprocess
import shlex
from datetime import datetime

ROOT = Path.cwd()
print("Workspace:", ROOT)
assert (ROOT / "scripts").exists(), "Run notebook from /workspace/thesis-llm-alignment"

Workspace: /workspace/thesis-llm-alignment


In [3]:
# Smoke config (small and quick)
BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATASET_ID = "tatsu-lab/alpaca"

RAW_DPO_PAIRS = "data/dpo_pairs.smoke.raw.jsonl"
CLEAN_DPO_PAIRS = "data/dpo_pairs.smoke.clean.jsonl"
CLEAN_STATS = "logs/dpo_pairs_smoke_stats.json"

SFT_OUT = "outputs/qwen2.5-3b-sft-lora-smoke"
DPO_OUT = "outputs/qwen2.5-3b-sft-dpo-lora-smoke"
COMPARE_OUT = "results/compare_outputs_smoke.jsonl"

NUM_EXAMPLES_FOR_DPO = 64
MAX_STEPS_SFT = 20
MAX_STEPS_DPO = 20
BATCH_SIZE = 1
GRAD_ACCUM = 8
MAX_SEQ_LEN = 768
MAX_PROMPT_LEN_DPO = 384
LR_SFT = 2e-4
LR_DPO = 1e-5
BETA_DPO = 0.1

GEN_MAX_NEW_TOKENS_BUILD = 96
GEN_MAX_NEW_TOKENS_COMPARE = 120
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.9

SMOKE_PROMPTS = [
    "Explain RLHF in simple terms.",
    "Give 3 bullet points on why unit tests matter.",
    "Translate to English: 'Мне нужно подготовить отчет по экспериментам.'",
    "What is overfitting and how can we reduce it?",
]

print("Smoke config loaded")

Smoke config loaded


In [4]:
def run(cmd: str):
    print("\n$", cmd)
    subprocess.run(cmd, shell=True, check=True)

def q(x):
    return shlex.quote(str(x))

Path("logs").mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

## 1) Build raw DPO pairs

In [5]:
cmd_build_pairs = (
    "python scripts/build_dpo_pairs.py "
    f"--base_model_id {q(BASE_MODEL_ID)} "
    f"--dataset_id {q(DATASET_ID)} "
    f"--out_path {q(RAW_DPO_PAIRS)} "
    f"--num_examples {NUM_EXAMPLES_FOR_DPO} "
    f"--max_new_tokens {GEN_MAX_NEW_TOKENS_BUILD} "
    f"--temperature {GEN_TEMPERATURE} "
    f"--top_p {GEN_TOP_P}"
)
run(cmd_build_pairs)


$ python scripts/build_dpo_pairs.py --base_model_id Qwen/Qwen2.5-3B-Instruct --dataset_id tatsu-lab/alpaca --out_path data/dpo_pairs.smoke.raw.jsonl --num_examples 64 --max_new_tokens 96 --temperature 0.7 --top_p 0.9
Loading dataset: tatsu-lab/alpaca


Generating train split: 100%|██████████| 52002/52002 [00:00<00:00, 283598.49 examples/s]


Using 64 examples for DPO pairs


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.89s/it]


Written 50 pairs
Done. Saved 63 pairs to data/dpo_pairs.smoke.raw.jsonl


## 2) Clean pairs

In [6]:
cmd_clean_pairs = (
    "python scripts/clean_dpo_pairs_full.py "
    f"--in_path {q(RAW_DPO_PAIRS)} "
    f"--out_path {q(CLEAN_DPO_PAIRS)} "
    "--min_chars 20 "
    "--drop_if_truncated "
    "--drop_if_equal "
    "--make_strict_prompt "
    f"--stats_path {q(CLEAN_STATS)}"
)
run(cmd_clean_pairs)


$ python scripts/clean_dpo_pairs_full.py --in_path data/dpo_pairs.smoke.raw.jsonl --out_path data/dpo_pairs.smoke.clean.jsonl --min_chars 20 --drop_if_truncated --drop_if_equal --make_strict_prompt --stats_path logs/dpo_pairs_smoke_stats.json
Done. kept=57, dropped=6, trimmed=6
Saved to: data/dpo_pairs.smoke.clean.jsonl
Stats saved to: logs/dpo_pairs_smoke_stats.json


## 3) Train SFT

In [7]:
cmd_train_sft = (
    "python scripts/train_sft.py "
    f"--model_id {q(BASE_MODEL_ID)} "
    f"--dataset_id {q(DATASET_ID)} "
    f"--output_dir {q(SFT_OUT)} "
    f"--max_steps {MAX_STEPS_SFT} "
    f"--lr {LR_SFT} "
    f"--batch_size {BATCH_SIZE} "
    f"--grad_accum {GRAD_ACCUM} "
    f"--max_seq_len {MAX_SEQ_LEN}"
)
run(cmd_train_sft)


$ python scripts/train_sft.py --model_id Qwen/Qwen2.5-3B-Instruct --dataset_id tatsu-lab/alpaca --output_dir outputs/qwen2.5-3b-sft-lora-smoke --max_steps 20 --lr 0.0002 --batch_size 1 --grad_accum 8 --max_seq_len 768
Loading dataset: tatsu-lab/alpaca
Loading tokenizer/model: Qwen/Qwen2.5-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.80s/it]
Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-commu

{'loss': 1.8861, 'grad_norm': 0.59765625, 'learning_rate': 0.00012454854871407994, 'entropy': 1.5459403648972512, 'num_tokens': 60602.0, 'mean_token_accuracy': 0.5933565150946378, 'epoch': 0.01}


100%|██████████| 20/20 [01:20<00:00,  3.85s/it]

{'loss': 1.4692, 'grad_norm': 0.376953125, 'learning_rate': 1.3638696597277679e-06, 'entropy': 1.5007646758109332, 'num_tokens': 119860.0, 'mean_token_accuracy': 0.6314452573657036, 'epoch': 0.03}


...

{'loss': 1.8962, 'grad_norm': 0.57421875, 'learning_rate': 0.00012454854871407994, 'entropy': 1.5548195272684098, 'num_tokens': 60602.0, 'mean_token_accuracy': 0.5921512592583895, 'epoch': 0.01}


100%|██████████| 20/20 [01:19<00:00,  4.04s/it]

{'loss': 1.4736, 'grad_norm': 0.3984375, 'learning_rate': 1.3638696597277679e-06, 'entropy': 1.509825759753585, 'num_tokens': 119860.0, 'mean_token_accuracy': 0.631589587777853, 'epoch': 0.03}


100%|██████████| 20/20 [01:22<00:00,  4.13s/it]


{'train_runtime': 82.6403, 'train_samples_per_second': 1.936, 'train_steps_per_second': 0.242, 'train_loss': 1.6849187374114991, 'epoch': 0.03}
Done. Saved to: outputs/qwen2.5-3b-sft-lora-smoke


## 4) Train DPO

In [8]:
cmd_train_dpo = (
    "python scripts/train_dpo.py "
    f"--base_model_id {q(BASE_MODEL_ID)} "
    f"--sft_adapter_dir {q(SFT_OUT)} "
    f"--dpo_data_path {q(CLEAN_DPO_PAIRS)} "
    f"--output_dir {q(DPO_OUT)} "
    f"--max_steps {MAX_STEPS_DPO} "
    f"--batch_size {BATCH_SIZE} "
    f"--grad_accum {GRAD_ACCUM} "
    f"--max_length {MAX_SEQ_LEN} "
    f"--max_prompt_length {MAX_PROMPT_LEN_DPO} "
    f"--lr {LR_DPO} "
    f"--beta {BETA_DPO}"
)
run(cmd_train_dpo)


$ python scripts/train_dpo.py --base_model_id Qwen/Qwen2.5-3B-Instruct --sft_adapter_dir outputs/qwen2.5-3b-sft-lora-smoke --dpo_data_path data/dpo_pairs.smoke.clean.jsonl --output_dir outputs/qwen2.5-3b-sft-dpo-lora-smoke --max_steps 20 --batch_size 1 --grad_accum 8 --max_length 768 --max_prompt_length 384 --lr 1e-05 --beta 0.1
Loading DPO pairs: data/dpo_pairs.smoke.clean.jsonl


Generating train split: 57 examples [00:00, 3876.31 examples/s]


Loading base model (4-bit): Qwen/Qwen2.5-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.68s/it]


Loading SFT adapter (trainable): outputs/qwen2.5-3b-sft-lora-smoke
Building reference model (frozen, same as SFT startpoint)


...

{'loss': 0.5773, 'grad_norm': 4.020435333251953, 'learning_rate': 6.227427435703997e-06, 'rewards/chosen': -0.0023901094682514668, 'rewards/rejected': -0.34824103116989136, 'rewards/accuracies': 0.6712328791618347, 'rewards/margins': 0.3458508551120758, 'logps/chosen': -88.67662811279297, 'logps/rejected': -112.97764587402344, 'logits/chosen': -0.9516248106956482, 'logits/rejected': -0.7755222320556641, 'epoch': 1.28}


100%|██████████| 20/20 [01:33<00:00,  4.76s/it]

{'loss': 0.3595, 'grad_norm': 3.965041399002075, 'learning_rate': 6.819348298638839e-08, 'rewards/chosen': -0.009291598573327065, 'rewards/rejected': -1.0470370054244995, 'rewards/accuracies': 0.8904109597206116, 'rewards/margins': 1.0377453565597534, 'logps/chosen': -86.90434265136719, 'logps/rejected': -115.7576904296875, 'logits/chosen': -0.8370327353477478, 'logits/rejected': -0.7187802195549011, 'epoch': 2.56}


100%|██████████| 20/20 [01:36<00:00,  4.84s/it]


{'train_runtime': 96.8561, 'train_samples_per_second': 1.652, 'train_steps_per_second': 0.206, 'train_loss': 0.4684098720550537, 'epoch': 2.56}
Done. Saved to: outputs/qwen2.5-3b-sft-dpo-lora-smoke


## 5) Compare outputs on a small prompt set

In [9]:
tmp_prompts_path = Path("data/compare_prompts.smoke.txt")
tmp_prompts_path.write_text("\n".join(SMOKE_PROMPTS) + "\n", encoding="utf-8")
print("Saved prompts to", tmp_prompts_path)

Saved prompts to data/compare_prompts.smoke.txt


In [10]:
cmd_compare = (
    "python scripts/compare_models.py "
    f"--base_model_id {q(BASE_MODEL_ID)} "
    f"--sft_adapter_dir {q(SFT_OUT)} "
    f"--dpo_adapter_dir {q(DPO_OUT)} "
    f"--prompts_path {q(tmp_prompts_path)} "
    f"--out_path {q(COMPARE_OUT)} "
    f"--max_new_tokens {GEN_MAX_NEW_TOKENS_COMPARE} "
    f"--temperature {GEN_TEMPERATURE} "
    f"--top_p {GEN_TOP_P} "
    "--do_sample"
)
run(cmd_compare)


$ python scripts/compare_models.py --base_model_id Qwen/Qwen2.5-3B-Instruct --sft_adapter_dir outputs/qwen2.5-3b-sft-lora-smoke --dpo_adapter_dir outputs/qwen2.5-3b-sft-dpo-lora-smoke --prompts_path data/compare_prompts.smoke.txt --out_path results/compare_outputs_smoke.jsonl --max_new_tokens 120 --temperature 0.7 --top_p 0.9 --do_sample
Loading tokenizer: Qwen/Qwen2.5-3B-Instruct
Loading base model (4-bit): Qwen/Qwen2.5-3B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.66s/it]


Loading SFT adapter: outputs/qwen2.5-3b-sft-lora-smoke
Reloading base model for DPO (4-bit): Qwen/Qwen2.5-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.64s/it]


Loading DPO adapter: outputs/qwen2.5-3b-sft-dpo-lora-smoke
Running 4 prompts. Saving to: results/compare_outputs_smoke.jsonl
  done 4/4
Done.


In [11]:
print("\nSmoke pipeline finished.")
print("Artifacts:")
print("-", RAW_DPO_PAIRS)
print("-", CLEAN_DPO_PAIRS)
print("-", CLEAN_STATS)
print("-", SFT_OUT)
print("-", DPO_OUT)
print("-", COMPARE_OUT)
print("Completed at:", datetime.now().isoformat(timespec="seconds"))


Smoke pipeline finished.
Artifacts:
- data/dpo_pairs.smoke.raw.jsonl
- data/dpo_pairs.smoke.clean.jsonl
- logs/dpo_pairs_smoke_stats.json
- outputs/qwen2.5-3b-sft-lora-smoke
- outputs/qwen2.5-3b-sft-dpo-lora-smoke
- results/compare_outputs_smoke.jsonl
Completed at: 2026-02-23T17:18:58
